# Lab 5 - Setup sources
Creates the lab5 schema and volume, checks the Netflix CSV and reads a few events from Event Hub.

In [ ]:
%run ./lab5_00_config

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.lab5_sources")
dbutils.fs.mkdirs(f"{volume_path}/netflix")
print("Upload dataset_example/netflix_titles.csv to:", source_csv_path)

Check for uploaded CSV

In [ ]:
titles = read_titles(spark, source_csv_path)
assert titles.count() == 8807, "Wrong file or CSV parsing configuration"
assert annotate_quality(clean_titles(titles), TITLE_RULES).filter("NOT _is_valid").count() == 0
assert titles.select("show_id").distinct().count() == 8807
print("PASS: Netflix CSV has 8,807 unique titles and no corrupt records")


## Event Hub check
Reads the events already stored in the hub, so run the producer first. It does not consume or delete anything.

In [ ]:
connection_string = dbutils.secrets.get(scope=secret_scope, key=secret_key)
options = kafka_options(namespace, eh_name, connection_string)
options.pop("maxOffsetsPerTrigger")
options["endingOffsets"] = "latest"
probe = decode_kafka(spark.read.format("kafka").options(**options).load())
rows = probe.select("kafka_topic", "kafka_partition", "kafka_offset", "kafka_enqueued_at").limit(5).collect()
assert rows, "Connection worked, but no retained events; run the producer and retry"
print("PASS: read retained events from Event Hub")
display(spark.createDataFrame(rows))